# VoiceGuard — AASIST-L Indic Fine-Tuning

Upload this single notebook to Colab. It performs setup, data validation, fine-tuning, test evaluation, and ONNX export without importing local `training/*.py` files. Enable **Runtime → Change runtime type → T4 GPU** before running it.

In [ ]:
!nvidia-smi
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.3.1', 'numpy==1.26.4', 'librosa==0.10.2.post1', 'soundfile==0.12.1', 'scikit-learn==1.5.1', 'pandas==2.2.2', 'tqdm==4.66.5', 'onnx==1.16.2', 'onnxruntime==1.18.1', 'huggingface-hub==0.24.7'], check=True)

## Data location
Mount Drive and set `DATA_ROOT`. It must contain `metadata.csv`, whose audio paths may be relative to that directory. Required columns: `clip_path,label,attack_family,generator,language,speaker_id,split`.

Labels are `bonafide` or `spoof`; splits are `train`, `dev`, and `test`. Each split needs both labels, speakers must never overlap across splits, and dev/test need an unseen spoof generator.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/voiceguard-data')  # Change if needed
METADATA_CSV = DATA_ROOT / 'metadata.csv'
OUTPUT_DIR = Path('/content/voiceguard-output')
UPSTREAM_DIR = Path('/content/aasist')
WEIGHTS_DIR = Path('/content/weights')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
assert METADATA_CSV.is_file(), f'Missing manifest: {METADATA_CSV}'
print(DATA_ROOT)

## Official AASIST-L source and initialization checkpoint
The model source is pinned to the reviewed ClovaAI revision. Its public AASIST-L checkpoint is used only for initialization; final metrics must come from your held-out Indic test split.

In [ ]:
import subprocess
from huggingface_hub import hf_hub_download

AASIST_REVISION = 'a04c9863f63d44471dde8a6abcb3b082b07cd1d1'
if not UPSTREAM_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/clovaai/aasist.git', str(UPSTREAM_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', AASIST_REVISION], cwd=UPSTREAM_DIR, check=True)
PRETRAINED = hf_hub_download(repo_id='SpeechAntiSpoofingBenchmarks/AASIST-L', filename='AASIST-L.pth', local_dir=str(WEIGHTS_DIR))
print('AASIST-L checkpoint:', PRETRAINED)

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

SEED = 42
SAMPLE_RATE = 16000
WINDOW_SAMPLES = 64600
BATCH_SIZE = 12
EPOCHS = 30
PATIENCE = 6
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 5e-6

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)

manifest = pd.read_csv(METADATA_CSV)
required = {'clip_path', 'label', 'attack_family', 'generator', 'language', 'speaker_id', 'split'}
assert not (required - set(manifest.columns)), f'Missing columns: {required - set(manifest.columns)}'
manifest.label = manifest.label.astype(str).str.lower().str.strip()
manifest.split = manifest.split.astype(str).str.lower().str.strip()
assert set(manifest.label) <= {'bonafide', 'spoof'}
assert set(manifest.split) <= {'train', 'dev', 'test'}
manifest.clip_path = manifest.clip_path.map(lambda x: str((DATA_ROOT / x).resolve()) if not Path(x).is_absolute() else str(Path(x)))
missing_audio = [x for x in manifest.clip_path if not Path(x).is_file()]
assert not missing_audio, f'Missing {len(missing_audio)} files; first: {missing_audio[0]}'
for split in ('train', 'dev', 'test'):
    assert set(manifest[manifest.split == split].label) == {'bonafide', 'spoof'}, f'{split} must contain both labels'
for first, second in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
    overlap = set(manifest[manifest.split == first].speaker_id) & set(manifest[manifest.split == second].speaker_id)
    assert not overlap, f'Speaker leakage {first}/{second}: {list(overlap)[:5]}'
def spoof_generators(frame):
    return set(frame[(frame.label == 'spoof') & ~frame.generator.fillna('').astype(str).str.lower().isin(['', 'none', 'nan', 'null', '-'])].generator.astype(str).str.lower())
train_generators = spoof_generators(manifest[manifest.split == 'train'])
for split in ('dev', 'test'):
    assert spoof_generators(manifest[manifest.split == split]) - train_generators, f'{split} needs an unseen spoof generator'
print(manifest.groupby(['split', 'label', 'language']).size().to_string())

## AASIST-L data loader
The model consumes a raw 16 kHz mono waveform. Training uses random windows; development and test use deterministic centre windows. A 16 kHz→8 kHz→16 kHz round-trip supplies label-neutral narrowband augmentation.

In [ ]:
import librosa, soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader

def read_audio(path):
    wave, rate = sf.read(path, dtype='float32', always_2d=False)
    if wave.ndim > 1: wave = wave.mean(axis=1)
    if rate != SAMPLE_RATE: wave = librosa.resample(wave, orig_sr=rate, target_sr=SAMPLE_RATE)
    return np.asarray(wave, dtype=np.float32)

def make_window(wave, training):
    if len(wave) >= WINDOW_SAMPLES:
        start = random.randint(0, len(wave) - WINDOW_SAMPLES) if training else (len(wave) - WINDOW_SAMPLES) // 2
        return wave[start:start + WINDOW_SAMPLES].copy()
    return np.pad(wave, (0, WINDOW_SAMPLES - len(wave))).astype(np.float32)

class IndicSpoofDataset(Dataset):
    def __init__(self, frame, split, training=False):
        self.rows = frame[frame.split == split].reset_index(drop=True); self.training = training
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        wave = make_window(read_audio(row.clip_path), self.training)
        if self.training and random.random() < 0.4:
            wave = librosa.resample(librosa.resample(wave, orig_sr=16000, target_sr=8000), orig_sr=8000, target_sr=16000)
        return torch.from_numpy(wave.astype(np.float32)), torch.tensor(int(row.label == 'spoof'), dtype=torch.long)

## Fine-tune official AASIST-L
This trains every AASIST-L parameter. It selects the best checkpoint using **development EER**, never the final test set.

In [ ]:
import importlib, sys
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Enable a Colab GPU for the real fine-tuning run.'
set_seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
if str(UPSTREAM_DIR) not in sys.path: sys.path.insert(0, str(UPSTREAM_DIR))
OfficialAASIST = importlib.import_module('models.AASIST').Model
AASIST_L_CONFIG = {'architecture':'AASIST', 'nb_samp':64600, 'first_conv':128, 'filts':[70,[1,32],[32,32],[32,24],[24,24]], 'gat_dims':[24,32], 'pool_ratios':[0.4,0.5,0.7,0.5], 'temperatures':[2.0,2.0,100.0,100.0]}
model = OfficialAASIST(AASIST_L_CONFIG).cuda()
blob = torch.load(PRETRAINED, map_location='cpu')
state = blob.get('model', blob.get('state_dict', blob)) if isinstance(blob, dict) else blob
state = {key.removeprefix('module.'): value for key, value in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing and not unexpected, f'Checkpoint mismatch: missing={missing}, unexpected={unexpected}'
train_loader = DataLoader(IndicSpoofDataset(manifest, 'train', True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_loader = DataLoader(IndicSpoofDataset(manifest, 'dev'), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0], device='cuda'))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')

@torch.inference_mode()
def scores(loader):
    model.eval(); labels, output = [], []
    for wave, label in loader:
        _, logits = model(wave.cuda(non_blocking=True))
        output.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy()); labels.extend(label.numpy())
    return np.asarray(labels), np.asarray(output)

def equal_error_rate(labels, output):
    fpr, tpr, thresholds = roc_curve(labels, output); fnr = 1 - tpr
    index = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[index] + fnr[index]) / 2), float(thresholds[index])

best_eer, no_improvement = float('inf'), 0
BEST_CKPT = OUTPUT_DIR / 'aasist-l-indic-best.pt'
for epoch in range(1, EPOCHS + 1):
    model.train(); loss_sum = 0.0
    for wave, label in tqdm(train_loader, desc=f'Epoch {epoch}'):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda'):
            _, logits = model(wave.cuda(non_blocking=True))
            loss = criterion(logits, label.cuda(non_blocking=True))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update(); loss_sum += loss.item()
    scheduler.step()
    dev_y, dev_score = scores(dev_loader); dev_eer, _ = equal_error_rate(dev_y, dev_score)
    print(f'epoch={epoch} train_loss={loss_sum / len(train_loader):.4f} dev_eer={dev_eer:.4f}')
    if dev_eer < best_eer:
        best_eer, no_improvement = dev_eer, 0
        torch.save({'model': model.state_dict(), 'config': AASIST_L_CONFIG, 'dev_eer': dev_eer}, BEST_CKPT)
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE: break
print('Best development EER:', best_eer, 'checkpoint:', BEST_CKPT)

## Evaluate final test and export ONNX
The threshold is derived from development data. The test set is evaluated once after model selection. The exported `spoof_logit` is positive for more spoof-like audio and is compatible with the VoiceGuard backend.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

checkpoint = torch.load(BEST_CKPT, map_location='cpu')
model.load_state_dict(checkpoint['model']); model.cuda().eval()
dev_y, dev_score = scores(dev_loader); _, threshold = equal_error_rate(dev_y, dev_score)
test_loader = DataLoader(IndicSpoofDataset(manifest, 'test'), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_y, test_score = scores(test_loader); test_eer, _ = equal_error_rate(test_y, test_score)
print(f'dev_threshold={threshold:.5f}; test_EER={test_eer:.4f}; ROC-AUC={roc_auc_score(test_y, test_score):.4f}; PR-AUC={average_precision_score(test_y, test_score):.4f}')
test_rows = manifest[manifest.split == 'test'].reset_index(drop=True).copy(); test_rows['score'] = test_score; test_rows['prediction'] = (test_rows.score >= threshold).astype(int)
for language, group in test_rows.groupby('language'):
    bona, spoof = group.label == 'bonafide', group.label == 'spoof'
    fpr = (group.loc[bona, 'prediction'] == 1).mean() if bona.any() else float('nan')
    fnr = (group.loc[spoof, 'prediction'] == 0).mean() if spoof.any() else float('nan')
    print(f'{language}: n={len(group)} FPR={fpr:.3f} FNR={fnr:.3f}')

class SpoofLogit(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, waveform):
        _, logits = self.base(waveform)
        return logits[:, 1] - logits[:, 0]

ONNX_PATH = OUTPUT_DIR / 'aasist-l-indic-v0.1.onnx'
export_model = SpoofLogit(model).eval()
example = torch.zeros(1, WINDOW_SAMPLES, device='cuda')
torch.onnx.export(export_model, example, str(ONNX_PATH), input_names=['waveform'], output_names=['spoof_logit'], dynamic_axes={'waveform': {0: 'batch'}, 'spoof_logit': {0: 'batch'}}, opset_version=17)
print('Exported:', ONNX_PATH)

In [ ]:
from google.colab import files
files.download(str(BEST_CKPT))
files.download(str(ONNX_PATH))